In [ ]:
import cv2
import numpy as np
from astropy.io import fits
from astroquery.gaia import Gaia
from astropy.coordinates.sky_coordinate import SkyCoord
from astropy.wcs import WCS
import sep
sep.set_extract_pixstack(1000000)
sep.set_sub_object_limit(1000000)

from astropy.table import Table, hstack
from scipy.optimize import curve_fit
from scipy.optimize import root_scalar
import csv
import math 
from scipy.stats import mode as spmode

import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground
from photutils.background import *
from photutils.datasets import make_100gaussians_image
from astropy.stats import biweight_location
from astropy.stats import mad_std
from astropy.stats import sigma_clipped_stats

import astropy.table
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from astropy.wcs import WCS
from astropy.table import Table, hstack

import os
from astropy.io import ascii
import glob
from glob import glob

from astroquery.astrometry_net import AstrometryNet

# sq func used in analysis 
def sq(x,a,b):
    return(a * (np.sign(x) * (np.abs(x))) **(b/2) )
    
import pandas as pd

mean = np.mean
std = np.std


# for handling arrays; image data are arrays!
import numpy as np 
# for plotting and displaying data
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
# for making lists of filenames
from glob import glob 
# for reading and writing FITS format data
from astropy.io import fits 
# for doing aperture photometry:
import photutils
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.stats import mad_std
# Import a function for fitting data
from scipy.optimize import curve_fit
# need to manually trigger garbage collector to free memory
import gc
from IPython.display import display, clear_output
import scipy.optimize as optimize


############### Set the platescale for your observation ##################

#platescale = 0.36 # arcsec / pixel    #ProEM Camera on the McDonald 82" telescope
#platescale = 0.964016 # arcsec / pixel   #CMOS on the Yerkes 41" telescope
platescale = 0.6 # arcsec / pixel    #CCD on the Yekres 24" telescope

#platescale=        # place to put platescales for other telescopes



%matplotlib widget

# for handling arrays; image data are arrays!
import numpy as np 
# for plotting and displaying data
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
# for making lists of filenames
from glob import glob 
# for reading and writing FITS format data
from astropy.io import fits 
# for doing aperture photometry:
import photutils
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.stats import mad_std
# Import a function for fitting data
from scipy.optimize import curve_fit
# need to manually trigger garbage collector to free memory
import gc
from IPython.display import display, clear_output



############### Set the platescale for your observation ##################

#platescale = 0.36 # arcsec / pixel    #ProEM Camera on the McDonald 82" telescope
#platescale = 0.964016 # arcsec / pixel   #CMOS on the Yerkes 41" telescope
platescale = 0.6 # arcsec / pixel    #CCD on the Yekres 24" telescope

#platescale=        # place to put platescales for other telescopes


import cv2
import numpy as np
from astropy.io import fits
from astroquery.gaia import Gaia
from astropy.coordinates.sky_coordinate import SkyCoord
from astropy.wcs import WCS
import sep
sep.set_extract_pixstack(1000000)
sep.set_sub_object_limit(1000000)

from astropy.table import Table, hstack
from scipy.optimize import curve_fit
import csv
import math 
from scipy.stats import mode as spmode

import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground
from photutils.background import *
from photutils.datasets import make_100gaussians_image
from astropy.stats import biweight_location
from astropy.stats import mad_std
from astropy.stats import sigma_clipped_stats

import astropy.table
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from astropy.wcs import WCS
from astropy.table import Table, hstack

import os
from astropy.io import ascii
import glob
from glob import glob

from astroquery.astrometry_net import AstrometryNet

# sq func used in analysis 
def sq(x,a,b):
    return(a * (np.sign(x) * (np.abs(x))) **(b/2) )
    
import pandas as pd

mean = np.mean
std = np.std


# for handling arrays; image data are arrays!
import numpy as np 
# for plotting and displaying data
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
# for making lists of filenames
from glob import glob 
# for reading and writing FITS format data
from astropy.io import fits 
# for doing aperture photometry:
import photutils
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.stats import mad_std
# Import a function for fitting data
from scipy.optimize import curve_fit
# need to manually trigger garbage collector to free memory
import gc
from IPython.display import display, clear_output
import scipy.optimize as optimize
from pathlib import Path
from datetime import datetime
from astropy.time import Time
import matplotlib.dates as mdates


############### Set the platescale for your observation ##################

#platescale = 0.36 # arcsec / pixel    #ProEM Camera on the McDonald 82" telescope
#platescale = 0.964016 # arcsec / pixel   #CMOS on the Yerkes 41" telescope
platescale = 0.6 # arcsec / pixel    #CCD on the Yekres 24" telescope

#platescale=        # place to put platescales for other telescopes

In [ ]:




def gaia(header):
    wcs_header = WCS(header)
    bottom_coord = wcs_header.pixel_to_world(0, 0)
    top_coord = wcs_header.pixel_to_world(header['NAXIS1'], header['NAXIS2'])
    full_coord_ra = np.abs(top_coord.ra.deg-bottom_coord.ra.deg)
    full_coord_dec = np.abs(top_coord.dec.deg-bottom_coord.dec.deg)
    ra_center = (top_coord.ra.deg + bottom_coord.ra.deg) / 2
    dec_center = (top_coord.dec.deg + bottom_coord.dec.deg) / 2
    job = Gaia.launch_job("SELECT TOP 300000 "
                    "source_id,ra,dec,parallax,parallax_error,pm,pmra,pmra_error,pmdec,pmdec_error,"
                    "phot_g_mean_mag, phot_g_mean_flux, phot_bp_mean_mag,phot_bp_mean_flux,phot_rp_mean_mag,"
                    "phot_rp_mean_flux,bp_rp, phot_variable_flag, classprob_dsc_combmod_galaxy"
                    " from gaiadr3.gaia_source"
                    " WHERE CONTAINS(POINT('ICRS',ra,dec),BOX('ICRS',{0},{1},{2},{3}))=1 AND"
                    "(phot_bp_mean_mag <= 18.0)".format(ra_center,
                                                                      dec_center,
                                                                    full_coord_ra, full_coord_dec))
    gaia_res = job.get_results()
    gaia_coord = SkyCoord(gaia_res['ra'], gaia_res['dec'], frame = 'icrs', unit = 'deg')

    return gaia_res, gaia_coord




def match_table(object_table, wcs, gaia_table, gaia_coordinates):
    pcor = wcs.pixel_to_world(object_table['x_centroid'], object_table['y_centroid'])
    object_table['ra_p'], object_table['dec_p'] = pcor.ra.deg, pcor.dec.deg
    index, d2d, ____ = pcor.match_to_catalog_sky(gaia_coordinates)
    mtab = hstack([gaia_table[index], Table(object_table)])
    _, unique_ind = np.unique(mtab['source_id'], return_index=True)
    mtab = mtab[unique_ind]
    mtab['ang_dist'] = np.abs(np.sqrt(mtab['ra']**2+mtab['dec']**2)
                              -np.sqrt(mtab['ra_p']**2+mtab['dec_p']**2))
    mtab['dec_res'] = (mtab['dec'] -  mtab['dec_p'])*3600
    mtab['ra_res'] = (mtab['ra'] -  mtab['ra_p'])*3600*0.9114

    return mtab







def zp(match, header, flux_name):
    zp_list = []
    exp = header['EXPTIME']
    match2 = match
    # if you want to filter out what stars it considers
    #tfilter = np.where(match['phot_bp_mean_mag'] <= 18) & (match['phot_variable_flag'] != 'VARIABLE')[0]
    #tfilter = np.where(match['phot_bp_mean_mag'] < 13) & (match['phot_bp_mean_mag'] >= 12)[0]
    #match2 = match[tfilter]


    for m in range(len(match2)):
        gaia_mag = match2['phot_bp_mean_mag'][m]
        flux = match2[flux_name][m]
        #print(gaia_mag)
        ##print(flux)
        #print(exp)
        zp = gaia_mag + 2.5*np.log10(flux/exp) # main equation
        zp_list.append(zp)

    avg_zp = np.nanmean(zp_list)
    return avg_zp



'''def zp(match, header, flux_name):

    exp = header['EXPTIME']

    match['zp'] = match['phot_bp_mean_mag'] + 2.5*np.log10(match[flux_name]/exp)

    return match
'''


def mag_comp(match, avg_zp, header, flux_name):
    exp = header['EXPTIME']
    match['new_mag'] = -2.5 * np.log10(match[flux_name]/exp) + avg_zp # main equation
    

    return match

def display(dat):
    plt.figure() #Create new "figure" for this image
    plt.imshow(dat,cmap='gray',vmin=np.percentile(data,5),
           vmax=np.percentile(data,98), origin = 'lower')
    plt.show()



def bkg_plots(data, bkg):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
    fig.set_figheight(8)
    fig.set_figwidth(15)
    p1 = ax1.imshow(bkg.background, origin='lower', cmap='Greys_r', interpolation='nearest')
    plt.colorbar(p1, ax = ax1, shrink=0.4)
    ax1.set_title('Bkg')
    p2 = ax2.imshow(bkg.background_rms, origin='lower', cmap='Greys_r', interpolation='nearest')
    plt.colorbar(p2, ax = ax2, shrink=0.4)
    ax2.set_title('RMS\n Med: %.4f   STD: %.4f'%(bkg.background_rms_median, std(bkg.background_rms)))
    ax3.imshow(data,cmap='gray',vmin=np.percentile(data,5),vmax=np.percentile(data,98), origin = 'lower')
    bkg.plot_meshes(outlines=True, marker='.', color='cyan', alpha=0.3)
    plt.show()







def source_detec(firstframe):
    bkg_sigma = mad_std(firstframe)
    daofind = DAOStarFinder(fwhm=3.5, threshold=5*bkg_sigma)
    sources = daofind(firstframe)
    return sources

def distance(x1,y1,x2,y2):
        return np.sqrt((x1-x2)**2.+(y1-y2)**2.)
    

def newap(firstframe, sources):
    bkgmed = np.median(data.flatten())
    r_base = sources['r']
    numsources = len(sources)
    positions = [(x,y) for x,y in sources['x_centroid','y_centroid']]

    starsum_list = []
    starapsum_list = []
    skysum_list = []
    averagesky_list = []

    for w in range(numsources):

        if np.isnan(r_base[w]) == True:
            starsum_list.append(np.nan)
            starapsum_list.append(np.nan)
            skysum_list.append(np.nan)
            averagesky_list.append(np.nan)

        else:
            
            # Define an example circular aperture of radius
            ap = CircularAperture(positions[w],r_base[w])
            # Define sky annulus in range 3 pixel radius
            
            rin = r_base[w] +5
            rout = rin + 4
            skyan = CircularAnnulus(positions[w],r_in=rin,r_out=rout)
    
            # Sum the pixel values in each aperture (target and comparison) and sky annuli
            starapsum = ap.do_photometry(firstframe-bkgmed)[0]
            skysum = skyan.do_photometry(firstframe-bkgmed)[0]
        
            averagesky = skysum/skyan.area
        
            # Then use this average to subtract the expected contribution from the sky
            # to the star apertures:
        
            starsum = starapsum - ap.area*averagesky
    
            starsum_list.append(starsum[0])
            starapsum_list.append(starapsum[0])
            skysum_list.append(skysum[0])
            averagesky_list.append(averagesky[0])



    sources['starsum'] = starsum_list
    sources['starapsum'] = starapsum_list
    sources['skysum'] = skysum_list
    sources['averagesky'] = averagesky_list

    
    newsources = sources[np.where(sources['starsum']>0)[0]]
    

    return newsources






# Visualize the target star profile.
%matplotlib inline

# Define the Gaussian function for fitting the stellar profile
def gaussian(x, A, sigma):
    # Definition of a Gaussian where:
    # - x is distance from center
    # - A is amplitude
    # - sigma is stddev
    # (FWHM = 2.3548*sigma)
    return A*np.exp(-(x)**2/(2.*sigma**2))


# Define function that returns the distance, in pixels, between two points:
def distance(x1,y1,x2,y2):
    return np.sqrt((x1-x2)**2.+(y1-y2)**2.)


# Second derivative of a Gaussian function
def gaussian_fprime2(x, A, mu, sigma):
    diff = x - mu
    term = (diff**2 - sigma**2) / (sigma**4)
    return A * np.exp(- (diff**2) / (2 * sigma**2)) * term



def determine_radius(data, otab):


    radius_list = []
    for s in range(len(otab)):
        # Plot the pixel values in terms of distance from the
        # center of the first star for pixels within ~15 of the pixels
        
        x0 = otab['x_centroid'][s]# x center of target star
        
        y0 = otab['y_centroid'][s] # y center of target star
        pdist = 15 #distance in pixels to examine
        dist = [] #empty list to hold pixel distances
        pval = [] #empty list to hold pixel values


        if round(x0)-pdist <0 or round(y0)-pdist<0:
            radius_list.append(np.nan)
            
        elif round(x0)+pdist > data.shape[0] or round(y0)+pdist > data.shape[1]:
            radius_list.append(np.nan)
            
        else: 

            
            for x in np.arange(round(x0)-pdist,round(x0)+pdist):
                for y in np.arange(round(y0)-pdist,round(y0)+pdist):
                    dist.append(distance(x,y,x0,y0))
                    # It bothers me that the image is indexed as (y,x)
                    pval.append(data[int(y),int(x)])
            
            
            
            '''# Plot the results (in units of pixels) in a new figure
            fig, ax1 = plt.subplots(num=3, sharey='row')
            ax1.scatter(np.array(dist),pval,c='gray')
            ax1.set_xlim(0,pdist)
            ax1.set_xlabel('distance (pixels)')
            ax1.set_ylabel('pixel values')
            #Make a second x-axis with arcsec units
            ax2 = ax1.twiny()
            ax2.set_xlabel('distance (arcsec)')
            ax2.set_xlim(0,pdist*platescale)'''
            
            
            # Find a Gaussian curve of best fit to the observed stellar
            # profile above the median image background (the scipy.optimize 
            # functions are great for fitting data!)
            
            # "Typical background"
            bkgmed = np.median(data.flatten())
            
            
            ########################### Set initial guess for peak pixel value and sigma ##################################
            peak = otab['peak'][s]  
            p0=[peak,1.] #initial guesses of height and sigma
            popt,_  = curve_fit(gaussian,dist,np.array(pval)-bkgmed,p0=p0)
            fwhm = np.abs(popt[-1])*2.3548*platescale
            
            
            # Plot this best fit
            gsamplespacing = 0.1
            gsample=np.arange(0,pdist+gsamplespacing,gsamplespacing)
            '''ax2.plot(gsample*platescale,bkgmed+
                     gaussian(gsample,popt[0],popt[1]),c='r',lw=3)
            # And show the HWHM
            ax2.plot([0,fwhm/2.,fwhm/2.],[bkgmed+(popt[0]/2.),bkgmed+(popt[0]/2.),bkgmed],
                     c='black',ls='-',lw=2)
            ax2.annotate('FWHM: {:.2f}"'.format(fwhm), xy=(fwhm/2.,bkgmed+popt[0]/2.), 
                         xycoords='data',xytext=(0.57,0.5), textcoords='axes fraction',
                         arrowprops=dict(facecolor='black', shrink=0.05,width=1),
                         ha='center',va='center',fontsize=18)
            
            # Set y-limits to show gaussian fit well
            ymax = (popt[0] + bkgmed )
            ax2.set_ylim(0,ymax*1.1)
            ax1.set_ylim(0,ymax*1.1)
            plt.show()'''
            
            
            # Define Gaussian parameters
            A = popt[0]    # Amplitude
            mu = np.mean(gsample)   # Mean
            sigma = popt[1] # Standard deviation
        
            # Use root_scalar to find where the second derivative is 0
            # We bracket the root between the mean (mu) and mu + 2*sigma
            sol = root_scalar(
                gaussian_fprime2, 
                args=(A, mu, sigma), 
                bracket=[mu, mu + 2 * sigma], 
                method='brentq')
            
            
            rad = sol.root - (sol.root/(fwhm+1))
            radius_list.append(rad)
        
    
    
    otab['r'] = radius_list
    
    return otab
    




    


In [ ]:
folders = glob('/raid6/users/rantoine/2026*')
print(folders)

In [ ]:
day = '2026-07-28'
path = '/raid6/users/rantoine/' + day + '/lights/*/*_c/'

files = glob(path+'*60s_B_c.new')



print(files)
print(len(files))


In [ ]:
%matplotlib inline
with fits.open(files[0]) as hdu:
    header = hdu[0].header

# gaia query
gtab, gcor = gaia(header)


mag_list = []
mag_list_2 = []
for f in range(len((files))):
    file = files[f]
    print('analyzing ' + str(file))
    with fits.open(file) as hdu:
        data = hdu[0].data.astype(np.float32)
        header = hdu[0].header

    #display(data)

    wcs= WCS(header)

    # source detection
    otab = source_detec(data)

    otab2 = determine_radius(data, otab)
    otab3 = newap(data, otab2)

    mtab = match_table(otab3, wcs, gtab,gcor)

    # calculate an average zero point magnitude for all sources
    avg_zp = zp(mtab, header, flux_name = 'starsum')
    print('avg zero point: ', str(avg_zp))
    
    # calculate magnitudes with zero point
    mtab2 = mag_comp(mtab, avg_zp, header, flux_name = 'starsum')

    # for target
    index = np.where(mtab2['source_id'] == 4307450193249412352)[0]
    print('target calibrated magnitude: ', str(mtab2['new_mag'][index][0]))

    # for standard
    index2 = np.where(mtab2['source_id'] == 4307450330692956928)[0]
    print('standard calibrated magnitude: ', str(mtab2['new_mag'][index2][0]))

    plt.figure() #Create new "figure" for this image
    plt.imshow(data,cmap='gray',vmin=np.percentile(data,5),
           vmax=np.percentile(data,98), origin = 'lower')
    plt.show()
    
    plt.figure()
    plt.scatter(mtab2['new_mag'], mtab2['phot_bp_mean_mag'], color = 'b', alpha = 0.4)
    plt.scatter(mtab2['new_mag'][index], mtab2['phot_bp_mean_mag'][index], color = 'r', alpha = 0.4)
    plt.scatter(mtab2['new_mag'][index2], mtab2['phot_bp_mean_mag'][index2], color = 'g', alpha = 0.4)
    #plt.scatter(mtab3['new_mag'], mtab3['phot_bp_mean_mag'], alpha = 0.4)
    #plt.plot(x_fit, y_fit, label=f'Fit: a={popt[0]:.2f}, b={popt[1]:.2f}')
    
    plt.xlabel('Calibrated Mag')
    plt.ylabel('Gaia mag')
    #plt.xlim(7,20)
    #plt.ylim(7,20)
    plt.show()

    plt.figure()
    plt.scatter(mtab2['bp_rp'], mtab2['new_mag'] - mtab2['phot_bp_mean_mag'], alpha = 0.4)
    plt.xlabel('Color')
    plt.ylabel('Delta Mag')
    plt.show()

    # for target
    mag_list.append(mtab2['new_mag'][index][0])

    # for standard
    mag_list_2.append(mtab2['new_mag'][index2][0])

    out_dir = Path('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/mag_list_output/')

    out_dir.mkdir(parents=True, exist_ok=True)

    # for target
    ####### CHANGE FILE NAME #########
    file_path = out_dir / '7-28-26_R_Aql_mag_list.fits'
    
    mag_data = np.array(mag_list, dtype=np.float32)
    hdu = fits.PrimaryHDU(data=mag_data)

    hdu.writeto(file_path, overwrite=True)

    # for standard
    ######## CHANGE FILE NAME #########

    file_path_2 = out_dir / '7-28-26_Standard_mag_list.fits'
    
    mag_data_2 = np.array(mag_list_2, dtype=np.float32)
    hdu_2 = fits.PrimaryHDU(data=mag_data_2)

    hdu_2.writeto(file_path_2, overwrite=True)


cleaned_list = [float(item) for item in mag_list]
#print(cleaned_list)
filtered_list = [x for x in cleaned_list if x <= 12]
#print(filtered_list)

cleaned_list_2 = [float(item) for item in mag_list_2]
#print(cleaned_list)
filtered_list_2 = [x for x in cleaned_list_2 if x <= 12]
#print(filtered_list)

print('mean = ', str(mean(filtered_list)))

print('std = ', str(std(filtered_list)))

print('mean = ', str(mean(filtered_list_2)))

print('std = ', str(std(filtered_list_2)))

In [ ]:
plt.close()
with fits.open('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/mag_list_output/7-28-26_R_Aql_mag_list.fits') as hdul:
    header = hdul[0].header
    data = hdul[0].data

with fits.open('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/mag_list_output/7-28-26_Standard_mag_list.fits') as hdul:
    header_2 = hdul[0].header
    data_2 = hdul[0].data

obs_num = range(len(data))

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(obs_num, data, color='r', s=10, label = 'R Aql')
ax.scatter(obs_num, data_2, color = 'b', s=10, label = 'Standard')
ax.set_xlabel('Observation')
ax.set_ylabel('Magnitude (BP)')
ax.invert_yaxis()

###### CHANGE TITLE #######
ax.set_title('7-28-26 24" Light Curve of R Aql and Standard Star')
plt.legend()
fig.autofmt_xdate()

###### CHNAGE NAME ########
plt.savefig('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/lc/7-28-26_light_curve.png')
plt.show()

In [ ]:
mag_path = '/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/mag_list_output/'

r_aql_mag_files = sorted(glob(mag_path+'*R_Aql_mag_list.fits'))
standard_mag_files = sorted(glob(mag_path+'*Standard_mag_list.fits'))

r_aql_dates = []
r_aql_avg_mag = []
for filepath in r_aql_mag_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_R_Aql_mag_list')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%m-%d-%y")
    #date = Time(date_time, scale="utc")
    
    with fits.open(filepath) as hdul:
        header = hdul[0].header
        mag = hdul[0].data
    cleaned_list = [float(item) for item in mag]
    filtered_list = [x for x in cleaned_list if x <= 12]
    r_aql_avg_mag.append(np.mean(filtered_list))
    r_aql_dates.append(date_time)
    
print(r_aql_avg_mag)
print(r_aql_dates)

standard_dates = []
standard_avg_mag = []
for filepath in standard_mag_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_Standard_mag_list')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%m-%d-%y")
    #date = Time(date_time, scale="utc")
    
    with fits.open(filepath) as hdul:
        header = hdul[0].header
        mag = hdul[0].data
    cleaned_list = [float(item) for item in mag]
    filtered_list = [x for x in cleaned_list if x <= 13]
    standard_avg_mag.append(np.mean(filtered_list))
    standard_dates.append(date_time)
    
print(standard_avg_mag)
print(standard_dates)

In [ ]:
plt.close()


plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(r_aql_dates, r_aql_avg_mag, color='r', s=10, label = 'R Aql')
ax.scatter(standard_dates, standard_avg_mag, color = 'b', s=10, label = 'Standard')
ax.set_xlabel('Observation')
ax.set_ylabel('New Magnitude (BP)')
ax.invert_yaxis()
ax.set_title('24" Light Curve of R Aql and Standard Star')


plt.legend()
fig.autofmt_xdate()
plt.savefig('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/lc/summer_26_light_curve.png')
plt.show()